# RSICD Adapter-CLIP — Session 5 of 5 — Ablations + Figures

Auto-generated from the rsicd-clip-adapter repo. Source of truth: `KAGGLE_RUNBOOK.md`.

**Before running this notebook:**
1. Click **+ Add data** in the right panel
2. Search for and add:
   - `thedevastator/rsicd-image-caption-dataset` (the original CSV — only needed for session 1)
   - Any `rsicd-*` Datasets you saved in previous sessions (e.g. `rsicd-adapter-s1`)
3. Settings: **Accelerator = GPU P100 or T4**, **Internet = ON**
4. Click **Save Version → Save Output** at the end of this session

In [ ]:
# === Install + clone + env ===
!pip install open_clip_torch faiss-cpu ftfy accelerate pyyaml -q
!git clone https://github.com/Vatsal057/rsicd-clip-adapter.git
%cd rsicd-clip-adapter
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("Repo cloned, deps installed.")

In [ ]:
# === Sanity: GPU + dataset location ===
import torch
from pathlib import Path
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
print()
data_path = Path("/kaggle/input/rsicd-image-caption-dataset")
if data_path.exists():
    print(f"Dataset:  {data_path}")
    csvs = sorted(p.name for p in data_path.glob('*.csv'))
    print(f"  CSVs:   {csvs}")
else:
    print(f"WARN: {data_path} does not exist.")
    print("   -> Click '+ Add data' in the right panel.")
    print("   -> Search for 'rsicd-image-caption-dataset' (the thedevastator version).")
    print("   -> Click 'Add' to attach it to this notebook.")
    print()
    # Show what IS attached, so the user can spot a typo in the dataset name
    print("Currently attached under /kaggle/input/:")
    try:
        attached = sorted(p.name for p in Path('/kaggle/input').iterdir())
    except FileNotFoundError:
        print("   (no /kaggle/input/ directory — running outside Kaggle?)")
        attached = []
    for name in attached:
        print(f"   - {name}")
    if not any('rsicd' in n.lower() for n in attached):
        print()
        print("   -> No RSICD-looking dataset found. The cell above is the fix:")
        print("      click '+ Add data', search 'rsicd-image-caption-dataset', and Add.")
    raise SystemExit(0)  # Stop here so the user can fix and re-run

## Required: attach the previous sessions' Datasets
Click **+ Add data** → search and add:
- `rsicd-adapter-s3` (for the adapter checkpoint used by figures)
- `rsicd-fullfinetune` (for the full-fine-tune metrics used by figures)

In [ ]:
# === Restore checkpoint from previous session ===
import shutil, os
from pathlib import Path
src = Path("/kaggle/input/rsicd-adapter-s3/adapter_best.pt")
if not src.exists():
    print(f"WARN: {src} not found. Did you forget to '+ Add data' rsicd-adapter-s3?")
else:
    Path("results/checkpoints").mkdir(parents=True, exist_ok=True)
    shutil.copy(src, "results/checkpoints/adapter_best.pt")
    print(f"Restored: {src}")
    # Also restore training history if it was saved
    src_h = Path("/kaggle/input/rsicd-adapter-s3/training_history_adapter.json")
    if src_h.exists():
        Path("results/metrics").mkdir(parents=True, exist_ok=True)
        shutil.copy(src_h, "results/metrics/training_history_adapter.json")
        print("Restored training history.")

If you want to split this session, run cells one at a time. Each ablation writes its own JSON when it finishes, so a mid-run disconnect loses at most one ablation.

In [ ]:
# === Ablations (placement + hidden_dim + data_size + residual) ===
# This runs ~20 hours on P100. If you hit the 12 h Kaggle limit, save what you have
# and resume in another notebook. Each ablation writes its own JSON when it finishes.
!python scripts/05_ablations.py

In [ ]:
# === Regenerate 4 paper figures with real data ===
!python scripts/06_qualitative.py

In [ ]:
# === Package output for next session ===
# After this cell, click 'Save Version' (top right) with 'Save Output' enabled.
# Then go to the Output tab -> 'New Dataset' -> name it 'rsicd-final'.
# The next session will attach this Dataset via '+ Add data'.
import shutil, os
out_dir = "/kaggle/working/rsicd-final"
os.makedirs(out_dir, exist_ok=True)
src = "results"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/figures"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/main.tex"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
src = "paper/refs.bib"
if os.path.exists(src):
    dst = os.path.join(out_dir, os.path.basename(src)) if not os.path.isdir(src) else out_dir
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(out_dir, os.path.basename(src)), dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
    print(f"  + {src} -> {out_dir}")
print(f"\nReady. Save this notebook version with 'Save Output' ON, then convert output to Dataset 'rsicd-final'.")

## Done!
Save the version, then convert output to Dataset `rsicd-final`.

**This is the last session.** Download `rsicd-final` from Kaggle → merge into your local repo → fill in `[TBD]` in `main.tex` → push to GitHub.